# 05. Gradio Chatbot 데모 배포 — HF Spaces + Langfuse


> **시나리오 안내** — 이 실습은 가상의 이커머스 마켓플레이스 **쇼핑몰**(자체 PB 라인: 베이직 · 프리미엄 · 라이트 · 한정판)를 소재로 합니다.

| 항목 | 내용 |
|------|------|
| 목적 | 04에서 양자화한 GGUF Q4_K_M을 Gradio로 감싸 **누구나 브라우저에서 써볼 수 있는 공개 데모**로 배포 |
| 서빙 모델 | 04에서 배포한 `<user>/product-cs-qwen3.5-2b-gguf` 중 `product-cs-q4_k_m.gguf` (~1.4GB) |
| 추론 엔진 | `llama-cpp-python` (CPU 추론, 04와 동일) |
| UI 프레임워크 | **Gradio** `ChatInterface` + System Prompt 토글 + 모델 선택 + 예시 질문 |
| 배포 위치 | **HF Spaces** (만료 없는 상시 공개 URL, 무료 `cpu-basic` 티어) |
| 모니터링 | Langfuse Cloud (**03/04와 동일 Project** 재사용) |
| 런타임 | Colab **CPU 런타임** (GPU 사용 안 함) — 14강 "GGUF가 CPU에서 돌아가는 이유" 실증 |
| 예상 비용 | $0 (CPU 추론 + HF Spaces 무료 `cpu-basic` 티어) |

> **사전 조건**
> - 04에서 `<user>/product-cs-qwen3.5-2b-gguf` repo에 GGUF 3종이 업로드돼 있어야 합니다.
> - 03에서 만든 Langfuse Project 키(`LANGFUSE_PUBLIC_KEY` / `SECRET_KEY`)가 필요합니다.
> - HF 토큰은 **Write 권한**이어야 합니다 (Spaces repo 생성 + 파일 업로드).

## 왜 데모 URL이 필요한가?

04까지는 `.gguf` 파일이 HF Hub에 올라가 있을 뿐입니다. 이 상태로는 **코드를 아는 개발자만** 그 모델을 써볼 수 있습니다. 비개발자(팀원·임원·사업부 동료)가 "이거 한번 써보고 싶은데?" 하면 파일 다운로드 → 환경 설치 → 명령어 입력이 필요합니다. 사실상 못 씁니다.

**브라우저에서 URL 하나 클릭하면 바로 채팅할 수 있는 데모**가 있어야 비로소 "누구나 쓸 수 있는 배포"가 완성됩니다. 이 노트북은 그 마지막 한 단계를 채웁니다.

## 왜 Gradio인가?

- **FastAPI + React**: 프로덕션 웹 서비스용 조합. 데모 하나 띄우려고 프론트/백엔드를 따로 짜는 건 오버스펙.
- **Streamlit**: Gradio와 비슷하지만 HF Spaces 통합이 부족합니다.
- **Gradio**: HF Spaces에 네이티브로 통합 → **ML 모델 데모 외부 공유의 최단 경로**.

## 왜 Q4_K_M GGUF를 서빙하는가?

- FP16(02 결과물) = GPU 필수 → Spaces 유료 GPU 티어 필요 💸
- **Q4_K_M(04 결과물) = CPU OK** → Spaces 무료 `cpu-basic`(16GB RAM, 2 vCPU) 티어로 충분 ✅
- Q4_K_M(1.4GB) + llama-cpp-python 오버헤드(~500MB) = 16GB에 여유롭게 올라갑니다.
- 14강 "GGUF가 CPU에서 돌아가는 이유"(순수 C++, 블록 단위 양자화, SIMD, mmap)가 **무료 배포로 증명**되는 구간.

---
## Phase 1: 환경 설정 — 패키지 설치

이 실습은 **GPU가 필요 없습니다.** Colab 런타임을 **CPU로 두고 시작**하세요 (런타임 메뉴 → 런타임 유형 변경 → 하드웨어 가속기: None).

설치 대상:

- **`gradio`**: 채팅 UI 프레임워크
- **`llama-cpp-python`**: GGUF를 CPU에서 추론 (04와 동일)
- **`huggingface_hub`**: GGUF 다운로드 + Spaces repo 생성·업로드
- **`langfuse`**: 모든 대화를 Trace로 기록 (03/04 Project 재사용)

In [4]:
# 다른 패키지 먼저 설치 (gradio·langfuse·huggingface_hub은 PyPI wheel 정상)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "gradio", "langfuse", "huggingface_hub"], check=True)

# llama-cpp-python — Colab 환경에서 검증된 manylinux2014 wheel 직접 다운로드.
#
# 왜 직접 URL인가:
#   abetlen pip index의 cp312-cp312-linux_x86_64 wheel이 musllinux(Alpine)로 잘못 빌드돼
#   Colab glibc 환경에서 "libc.musl-x86_64.so.1: cannot open shared object" 에러 발생.
#   → 검증된 manylinux2014 wheel URL을 직접 명시해 이 함정을 회피.
#
# 버전 0.3.21: 04에서 GGUF 만든 버전과 동일 (호환 보장).
WHEEL_URL = (
    "https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.21/"
    "llama_cpp_python-0.3.21-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"
)

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "llama-cpp-python"],
               stderr=subprocess.DEVNULL)

# 1차: 검증된 manylinux2014 wheel 직접 설치
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", WHEEL_URL])

# 2차 fallback: URL이 미래에 사라지면 abetlen index → 컴파일 순서로 시도
if r.returncode != 0:
    print("[install] ⚠️ 1차 직접 URL 실패 → abetlen index로 fallback")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cpu",
        "llama-cpp-python",
    ], check=True)
    try:
        import importlib, llama_cpp
        importlib.reload(llama_cpp)
        llama_cpp.llama_supports_gpu_offload()
    except Exception as e:
        print(f"[install] ⚠️ abetlen wheel도 musllinux 문제 → 컴파일 fallback (5~10분)")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "llama-cpp-python"],
                       stderr=subprocess.DEVNULL)
        subprocess.run([sys.executable, "-m", "pip", "install", "--prefer-binary",
                        "llama-cpp-python==0.3.21"], check=True)

import gradio as gr
import importlib, llama_cpp
importlib.reload(llama_cpp)
import langfuse as _lf_mod

print(f"✅ gradio {gr.__version__}")
print(f"✅ llama-cpp-python {llama_cpp.__version__}  (GPU offload: {llama_cpp.llama_supports_gpu_offload()})")
print(f"✅ langfuse {_lf_mod.__version__}")


---
## Phase 2: 인증 — HF / Langfuse / (옵션) OpenAI

03/04에서 쓴 **동일한 키**를 그대로 입력합니다. Langfuse Project도 동일하게 재사용해서, 한 대시보드에서 "학습→평가→양자화→실 사용"을 누적해서 봅니다.

- **`HF_TOKEN`** — Write 권한 (Spaces repo 생성 + 파일 업로드에 필요)
- **`LANGFUSE_PUBLIC_KEY` / `SECRET_KEY`** — 03에서 만든 Project 것 그대로
- **`LANGFUSE_BASE_URL`** — `https://jp.cloud.langfuse.com` (Japan 리전, 한국에서 지연 최적)

In [5]:
import os
from getpass import getpass
from huggingface_hub import login

# (1) HF 토큰 — Write 권한 필요
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("HF_TOKEN (Write 권한): ").strip()
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

# (2) Langfuse Cloud — 03/04와 동일 Project
if not os.environ.get("LANGFUSE_PUBLIC_KEY"):
    os.environ["LANGFUSE_PUBLIC_KEY"] = getpass("LANGFUSE_PUBLIC_KEY (pk-lf-...): ").strip()
if not os.environ.get("LANGFUSE_SECRET_KEY"):
    os.environ["LANGFUSE_SECRET_KEY"] = getpass("LANGFUSE_SECRET_KEY (sk-lf-...): ").strip()
os.environ.setdefault("LANGFUSE_BASE_URL", "https://jp.cloud.langfuse.com")
os.environ["LANGFUSE_HOST"] = os.environ["LANGFUSE_BASE_URL"]

print("\n✅ 인증 완료")
print(f"   Langfuse host: {os.environ['LANGFUSE_BASE_URL']}")

---
## Phase 3: HF Hub에서 GGUF Q4_K_M 다운로드

04에서 올린 `<user>/product-cs-qwen3.5-2b-gguf` repo에서 **Q4_K_M 파일 하나만** 받습니다. FP16과 Q8_0은 이 실습에서 쓰지 않습니다.

- 파일명: `product-cs-q4_k_m.gguf`
- 예상 크기: ~1.4GB
- `hf_hub_download`는 **캐싱**을 해서 두 번째 실행부터는 즉시 로드됩니다.

In [6]:
from pathlib import Path
from huggingface_hub import hf_hub_download, whoami

HF_USERNAME = whoami()["name"]
GGUF_REPO   = f"{HF_USERNAME}/product-cs-qwen3.5-2b-gguf"
GGUF_FILE   = "product-cs-q4_k_m.gguf"

print(f"다운로드 대상: {GGUF_REPO}/{GGUF_FILE}\n")

MODEL_PATH = hf_hub_download(
    repo_id=GGUF_REPO,
    filename=GGUF_FILE,
    local_dir="./models",
    token=True,
)

size_gb = Path(MODEL_PATH).stat().st_size / 1e9
print(f"\n✅ 다운로드 완료: {MODEL_PATH}")
print(f"   파일 크기: {size_gb:.2f} GB  (예상: ~1.4GB)")

---
## Phase 4: llama-cpp-python으로 모델 로드 + 단일 추론

**`llama-cpp-python`이란?** llama.cpp(C++ LLM 추론 엔진)를 Python에서 부를 수 있게 감싼 라이브러리. 04 실습에서 쓴 것과 동일합니다.

**`Llama(...)` 주요 파라미터**

- **`n_ctx=2048`** — 한 번에 처리할 수 있는 최대 토큰 수 (대화 누적 길이 상한)
- **`n_threads=4`** — CPU 코어 수 (Colab CPU 런타임은 2~4코어)
- **`verbose=False`** — 매 추론마다 디버그 로그를 끕니다 (데모용이라 조용하게)

**확인 포인트**: 이 셀이 CPU 런타임에서 돌고 있다는 점입니다. **GPU 없이 LLM 답변이 나오는 것**이 14강의 핵심 증명입니다.

In [7]:
from llama_cpp import Llama

llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=2048,
    n_threads=4,
    verbose=False,
)

# --- 새너티 체크: 단일 추론 ---
test_q = "배터리가 빨리 닳아요. 어떻게 해야 하나요?"
out = llm.create_chat_completion(
    messages=[{"role": "user", "content": test_q}],
    max_tokens=256,
    temperature=0.7,
)
answer = out["choices"][0]["message"]["content"]

print("─" * 60)
print(f"질문: {test_q}")
print(f"답변: {answer}")
print("─" * 60)
print("\n💡 이 답변이 CPU만으로 나왔다는 점이 14강 핵심 포인트입니다.")

---
## Phase 5: Langfuse 초기화 (03/04 Project 재사용)

### 왜 "추론 시" Langfuse Trace가 중요한가?

03/04에서 Langfuse는 **"미리 준비한 질문으로 평가 점수를 기록하는 도구"**였습니다. 05에서는 역할이 달라집니다. **실 사용자가 예측 불가능한 질문을 던지는 환경**이기 때문입니다.

| Trace 없이 | Trace 있으면 |
|---|---|
| 사용자들이 뭘 물어보는지 모름 | 실 사용 패턴 분석 가능 |
| 품질 저하 케이스 발견 불가 | 느린 응답·이상한 답변 추적 → 재학습 근거 |
| 응답 시간 이슈 감지 못 함 | latency 분포 모니터링 |

03의 "평가 데이터"와 05의 "실 사용 데이터"가 **같은 Project에 공존**합니다 → 학습→평가→배포→모니터링 **MLOps 전체 루프**의 시작입니다.

### 04와 동일한 v4 API

- **`get_client()`** — 환경변수(`LANGFUSE_PUBLIC_KEY` 등)에서 자동으로 설정 로드
- **`propagate_attributes(tags=[...])`** — 이 컨텍스트 안에서 만들어지는 모든 observation에 태그 자동 부착
- **`start_observation(as_type="generation")`** + **`update/end`** — LLM 호출 한 건을 기록

In [8]:
from langfuse import get_client, propagate_attributes

langfuse = get_client()
print(f"✅ Langfuse 클라이언트 초기화 (SDK v{_lf_mod.__version__})")
print(f"   대시보드: {os.environ['LANGFUSE_BASE_URL']}")

# 대화 한 건 당 부착할 공통 태그. 대시보드에서 필터용.
DEMO_TAGS = ["gradio-demo", "production", "q4_k_m"]
CONDITION = "SFT+Q4_K_M+Gradio"

# CPU 추론 비용 환산 기준 (Colab CPU는 무료지만 Spaces cpu-basic 표준화를 위해 04와 동일 단가 사용)
ESTIMATED_CPU_COST_PER_HOUR_USD = 0.05

---
## Phase 6: 추론 함수 정의 — Gradio가 호출할 `chat_fn`

**동작 의도**

Gradio `ChatInterface`가 채팅 턴마다 부를 함수입니다. 한 번 호출될 때마다 **Langfuse observation 하나**가 자동으로 생성되고, 질문·답변·지연시간·토큰 수가 모두 기록됩니다.

**시그니처**

- **`message`** — 사용자가 방금 입력한 질문
- **`history`** — 이전까지의 대화 기록 (Gradio `type="messages"` 포맷)
- **`system_prompt`** — Accordion으로 토글 가능. 비워두면 **"SFT + No Prompt"**, 채워두면 **"SFT + Prompt"** (03 실습의 조건 3 vs 조건 4 라이브 체험)
- **`model_choice`** — **SFT(우리 모델)** vs **Base(학습 전 Qwen3-1.7B)** 선택. 같은 프롬프트·같은 질문을 두 모델에 던져 **SFT 각인의 효과를 UI에서 직접 비교**

**Base 모델 지연 로드**

Base(`unsloth/Qwen3-1.7B-GGUF`)는 **사용자가 최초 선택하는 순간에만** 다운로드(~1.1GB) + 로드됩니다. SFT 하나로만 쓰는 사람은 Base를 전혀 받지 않습니다. `_base_llm` 모듈 싱글톤으로 두 번째 선택부터는 캐시 재사용.

**Langfuse 기록 포인트**

- **`metadata.condition`** = `"SFT+Q4_K_M+Gradio"` → 03/04 condition과 같은 차원
- **`metadata.model_choice`** = `"SFT ..."` / `"Base ..."` → 대시보드에서 모델별 비교 필터
- **`metadata.system_prompt_used`** → System Prompt 토글 여부
- **`metadata.deployment`** → 실행 환경 구분 (`"colab"` / `"hf_spaces"`)
- **`tags`** → `gradio-demo` + 모델 태그(`qwen3.5-2b-sft-q4_k_m` / `qwen3-1.7b-base`)

In [9]:
import time

DEPLOYMENT = "colab"  # Colab에서 chat_fn을 직접 호출할 때의 실행 환경. Spaces app.py에서는 "hf_spaces"

# --- Base 모델 정보 (SFT 대조 실험용) ---
# 우리 SFT 학습 '이전' 상태의 근사치. 같은 Qwen3 계열의 공식 Instruct 버전.
BASE_GGUF_REPO = "unsloth/Qwen3-1.7B-GGUF"
BASE_GGUF_FILE = "Qwen3-1.7B-Q4_K_M.gguf"

# 모델 선택 라벨 상수 (Gradio Radio 값 ↔ chat_fn 분기 키로 공용)
SFT_CHOICE  = "SFT (우리 모델)"
BASE_CHOICE = "Base (Qwen3-1.7B, 학습 안 함)"

_base_llm = None  # Base 모델 지연 로드용 싱글톤


def ensure_base_llm():
    """사용자가 Base를 최초로 선택하는 순간에만 다운로드+로드. 이후 재사용."""
    global _base_llm
    if _base_llm is None:
        print("Base 모델 최초 로드 중 (~1분 소요)...")
        path = hf_hub_download(
            repo_id=BASE_GGUF_REPO,
            filename=BASE_GGUF_FILE,
            local_dir="./models-base",
        )
        _base_llm = Llama(
            model_path=path,
            n_ctx=2048,
            n_threads=4,
            verbose=False,
        )
        print("✅ Base 모델 로드 완료")
    return _base_llm


def chat_fn(message: str, history: list, system_prompt: str, model_choice: str = SFT_CHOICE):
    """Gradio ChatInterface가 호출할 함수. 한 턴 = 한 Langfuse observation."""
    # --- 0) 모델 선택 ---
    if model_choice == BASE_CHOICE:
        model = ensure_base_llm()
        model_tag = "qwen3-1.7b-base"
    else:
        model = llm
        model_tag = "qwen3.5-2b-sft-q4_k_m"

    # --- 1) 메시지 구성 ---
    messages = []
    if system_prompt and system_prompt.strip():
        messages.append({"role": "system", "content": system_prompt.strip()})
    for h in history:
        messages.append({"role": h["role"], "content": h["content"]})
    messages.append({"role": "user", "content": message})

    # --- 2) Langfuse 기록 시작 ---
    with propagate_attributes(tags=DEMO_TAGS + [model_tag]):
        generation = langfuse.start_observation(
            as_type="generation",
            name="gradio-demo-chat",
            model=model_tag,
            input=messages,
            metadata={
                "condition": CONDITION,
                "model_choice": model_choice,
                "system_prompt_used": bool(system_prompt and system_prompt.strip()),
                "deployment": DEPLOYMENT,
                "inference_engine": "llama.cpp",
                "question": message,
            },
        )

        # --- 3) 추론 ---
        t0 = time.perf_counter()
        res = model.create_chat_completion(
            messages=messages,
            max_tokens=512,
            temperature=0.7,
        )
        latency_s  = time.perf_counter() - t0
        latency_ms = int(latency_s * 1000)

        text = res["choices"][0]["message"]["content"]
        # Qwen <think> 블록 방어적 제거
        if "</think>" in text:
            text = text.split("</think>", 1)[1].lstrip()

        usage = res.get("usage", {}) or {}
        in_tok  = int(usage.get("prompt_tokens", 0))
        out_tok = int(usage.get("completion_tokens", 0))
        tot_tok = in_tok + out_tok
        tps     = out_tok / latency_s if latency_s > 0 else 0.0

        est_cost    = (latency_s / 3600.0) * ESTIMATED_CPU_COST_PER_HOUR_USD
        input_cost  = est_cost * (in_tok  / tot_tok) if tot_tok else 0.0
        output_cost = est_cost * (out_tok / tot_tok) if tot_tok else 0.0

        # --- 4) Langfuse 기록 종료 ---
        generation.update(
            output=text,
            usage_details={"input": in_tok, "output": out_tok, "total": tot_tok},
            cost_details={
                "input":  round(input_cost,  8),
                "output": round(output_cost, 8),
                "total":  round(est_cost,    8),
            },
            metadata={
                "latency_ms": latency_ms,
                "tokens_per_sec": round(tps, 2),
                "cost_method": "estimated_cpu_time",
            },
        )
        generation.end()

    return text


# --- 새너티 체크: Gradio 없이 SFT 경로 직접 호출 ---
sample = chat_fn(
    message="카메라 초기화하는 방법 알려주세요",
    history=[],
    system_prompt="당신은 쇼핑몰 고객지원 상담원입니다. 격식체로 단계별로 안내합니다.",
    model_choice=SFT_CHOICE,
)
print("─" * 60)
print(sample)
print("─" * 60)
print("\n💡 방금 이 대화 한 건이 Langfuse에 Trace로 쌓였습니다. (tag: gradio-demo)")

---
## Phase 7: Gradio UI 구성 (launch는 Spaces에서)

**핵심 결정: Colab에서 `launch`를 호출하지 않습니다.**

우리의 최종 목적지는 **HF Spaces의 만료 없는 상시 공개 URL**입니다. Colab은 세션이 끊기면 사라지는 임시 환경이라, UI를 굳이 Colab에서 띄울 이유가 없어요. 이미 Phase 6에서 `chat_fn`을 직접 호출해 답변이 정상적으로 생성되는 걸 확인했으니, UI wrapper 동작은 Spaces에서 검증하면 됩니다.

**따라서 이 셀에서는**:

1. `gr.ChatInterface` 객체를 정의해 **UI 구성이 Spaces에 올릴 `app.py`와 동일함을 확인**만 하고
2. `launch()`는 호출하지 않습니다 (Phase 8에서 Spaces 서버가 자동으로 올림)

**UI 구성 요소**

- **`gr.ChatInterface(fn=chat_fn, ...)`** — `chat_fn`을 채팅 UI로 감쌉니다
- **`title` / `description`** — 상단 헤더
- **`examples=[...]`** — 예시 질문 버튼
- **`additional_inputs=[Textbox(System Prompt), Radio(모델 선택)]`** — 프롬프트 토글 + 모델 선택
- **`type="messages"`** — OpenAI 호환 포맷

In [10]:
# --- 예시 질문 (이커머스 고객 지원 도메인, 카테고리 균형) ---
examples = [
    ["밤에 사진 찍으면 색이 이상해요"],
    ["폴드 펼칠 때 화면이 깜빡여요"],
    ["배터리가 하루도 안 가요"],
    ["카메라 초기화하는 방법 알려주세요"],
    ["스마트홈 액세서리에서 알림이 안 울려요"],
    ["화면 보호 필름 추천해주세요"],
]

DEFAULT_SYSTEM_PROMPT = (
    "당신은 쇼핑몰 고객지원 상담원입니다. "
    "친절하고 단계별로 안내합니다. 격식체(~합니다)를 사용합니다."
)

# --- ChatInterface를 '최상위'로 구성 (Phase 8 app.py와 동일 구조) ---
demo = gr.ChatInterface(
    fn=chat_fn,
    type="messages",
    title="쇼핑몰 고객 지원팀 고객 상담 챗봇",
    description=(
        "두 가지를 UI에서 직접 비교해볼 수 있는 창구입니다.\n\n"
        "1) **System Prompt 토글** — 03 평가 실습의 조건 3(SFT만) vs 조건 4(SFT + 프롬프트).\n"
        "2) **모델 선택** — 우리가 학습한 **SFT 모델** vs 학습 안 한 **Base(Qwen3-1.7B)**. "
        "같은 프롬프트를 두 모델에 던져보면 SFT 각인이 프롬프트를 어떻게 이기는지 직접 확인 가능.\n\n"
        "모든 턴은 Langfuse Trace로 남아 03·04 평가 데이터와 같은 Project에서 함께 조회됩니다."
    ),
    examples=examples,
    additional_inputs=[
        gr.Textbox(
            label="System Prompt",
            value=DEFAULT_SYSTEM_PROMPT,
            lines=3,
            info="비워두면 순수 SFT 모델로만 답변합니다.",
        ),
        gr.Radio(
            label="모델 선택",
            choices=[SFT_CHOICE, BASE_CHOICE],
            value=SFT_CHOICE,
            info="Base를 처음 선택하면 약 1분간 다운로드+로드가 일어납니다 (이후 즉시).",
        ),
    ],
    theme=gr.themes.Soft(),
)

# --- 실행 옵션 ---
# 기본: launch()를 호출하지 않음. 공개 URL은 Phase 8 HF Spaces가 자동으로 만들어 줍니다.
# (chat_fn은 Phase 6에서 직접 호출로 이미 정상 동작이 확인됐으므로 굳이 Colab에서 띄울 필요 없음)
#
# 단, Phase 8/9 Spaces 빌드가 실패하거나 빌드 5~10분이 부담스러울 때
# → 아래 USE_COLAB_PREVIEW = True 로 바꿔 이 셀을 다시 실행하면
#   Colab 셀 안에 UI가 임베드됨 (gradio.live 공개 URL은 만들지 않고 Colab 내부에서만 동작).
USE_COLAB_PREVIEW = True  # 문제 있을 때만 True로

if USE_COLAB_PREVIEW:
    print("⚙️  USE_COLAB_PREVIEW=True → Colab 셀에 UI 임베드합니다 (share=False, inline=True)")
    try:
        demo.launch(share=False, inline=True, debug=False)
    except Exception as e:
        print(f"⚠️  launch 실패: {e}")
        print("    → Spaces 배포(Phase 8)로 진행하거나 런타임 재시작 후 재시도.")
else:
    print("✅ ChatInterface 객체 정의 완료 (launch는 Phase 8 Spaces 배포에서 수행)")
    print(f"   - 예시 질문:  {len(examples)}개")
    print(f"   - 모델 선택:  {SFT_CHOICE} / {BASE_CHOICE}")
    print(f"   - 타입:       {demo.__class__.__name__}")
    print(f"   - 다음 셀(Phase 8)로 바로 진행하세요.")
    print(f"   - (Spaces 빌드가 안 풀리면 USE_COLAB_PREVIEW=True 로 Colab 미리보기 가능)\n")

---
## Phase 8: HF Spaces 배포용 파일 작성

지금까지 Colab에서 모델이 잘 도는 걸 확인했습니다. 이제 이 코드를 **HF Spaces**라는 무료 호스팅 서비스에 올려서 누구나 쓸 수 있는 공개 URL을 만들 거예요.

이 셀은 업로드할 파일 3개를 **로컬 `./spaces-app/` 폴더에 자동으로 만들어 줍니다**. 실제 업로드는 다음 셀에서.

**3개 파일이 만들어지는 이유**

- **`app.py`** — Spaces가 실행할 메인 스크립트. Phase 6의 `chat_fn` + Phase 7의 UI를 합친 것. **Colab에서 본 그대로** 동작.
- **`Dockerfile`** — Spaces가 모델·라이브러리를 자동 설치하는 레시피. **수정할 필요 없음**.
- **`README.md`** — Spaces 페이지 상단에 표시될 메타데이터.

> ⚠️ 이미 같은 이름 Space가 있으면 HF 웹에서 **먼저 삭제**하거나 `SPACE_REPO_ID`를 `-v2` 등으로 바꿔주세요.

In [11]:
from pathlib import Path

SPACES_DIR = Path("spaces-app")
SPACES_DIR.mkdir(exist_ok=True)

SPACE_REPO_ID = f"{HF_USERNAME}/product-cs-chatbot-demo"

# --- Dockerfile ---
# PYTHONUNBUFFERED=1 + app.py 내부 sys.stdout.reconfigure 조합으로
# Container 로그가 실시간 보장.
dockerfile = '''FROM python:3.11-slim

RUN apt-get update && \\
    apt-get install -y --no-install-recommends build-essential cmake && \\
    rm -rf /var/lib/apt/lists/*

ENV CMAKE_BUILD_PARALLEL_LEVEL=4
ENV CMAKE_ARGS="-DLLAMA_NATIVE=OFF"
ENV PYTHONUNBUFFERED=1

RUN pip install --no-cache-dir "llama-cpp-python==0.3.20" && \\
    pip install --no-cache-dir gradio==5.50.0 huggingface_hub langfuse

WORKDIR /app
COPY app.py .

EXPOSE 7860
CMD ["python", "-u", "app.py"]
'''
(SPACES_DIR / "Dockerfile").write_text(dockerfile, encoding="utf-8")

# --- app.py ---
# 로그 확실성 보강:
#  - python -u (Dockerfile CMD) 로 실행
#  - sys.stdout.reconfigure(line_buffering=True) 로 라인 버퍼링 강제
#  - logging 모듈도 병행 사용 (print 가 안 뜰 경우 대비)
#  - [boot]/[init]/[turn] 태그로 로그 추적 쉽게
app_py = f'''import os
import sys
import time
import logging

# ---- 로그 확실하게 찍기 ----
try:
    sys.stdout.reconfigure(line_buffering=True)
    sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    stream=sys.stdout,
)
log = logging.getLogger("gradio-demo")

def _say(msg: str) -> None:
    """print + logging 이중 송출. 어느 쪽이든 Container 로그에 나타남."""
    print(msg, flush=True)
    log.info(msg)


import gradio as gr
from huggingface_hub import hf_hub_download
from llama_cpp import Llama
from langfuse import get_client, propagate_attributes

_say("=" * 70)
_say(f"[boot] app.py 시작 | python={{sys.version.split()[0]}}")

# ---------- 상수 ----------
GGUF_REPO_ID  = "{GGUF_REPO}"
GGUF_FILENAME = "{GGUF_FILE}"

BASE_GGUF_REPO_ID  = "unsloth/Qwen3-1.7B-GGUF"
BASE_GGUF_FILENAME = "Qwen3-1.7B-Q4_K_M.gguf"

SFT_CHOICE  = "SFT (우리 모델)"
BASE_CHOICE = "Base (Qwen3-1.7B, 학습 안 함)"

DEFAULT_SYSTEM_PROMPT = (
    "당신은 쇼핑몰 고객지원 상담원입니다. "
    "친절하고 단계별로 안내합니다. 격식체(~합니다)를 사용합니다."
)

MAX_NEW_TOKENS = 512
GEN_STOP_TOKENS = ["<|im_end|>"]

DEMO_TAGS  = ["gradio-demo", "production", "q4_k_m"]
CONDITION  = "SFT+Q4_K_M+Gradio"
DEPLOYMENT = "hf_spaces"

# ---------- Langfuse ----------
os.environ.setdefault(
    "LANGFUSE_HOST",
    os.environ.get("LANGFUSE_BASE_URL", "https://jp.cloud.langfuse.com"),
)
try:
    langfuse = get_client()
    _say("[boot] Langfuse client: OK")
except Exception as e:
    langfuse = None
    _say(f"[boot] Langfuse client: SKIPPED ({{e}})")


# ---------- 모델 지연 초기화 ----------
_llm = None
_base_llm = None


def ensure_llm():
    global _llm
    if _llm is None:
        _say(f"[init] SFT 모델 다운로드 중: {{GGUF_REPO_ID}}/{{GGUF_FILENAME}}")
        p = hf_hub_download(repo_id=GGUF_REPO_ID, filename=GGUF_FILENAME)
        _say(f"[init] SFT 파일 다운로드 완료: {{p}}")
        _llm = Llama(model_path=p, n_ctx=2048, n_threads=2, n_batch=512, verbose=False)
        _say("[init] SFT 모델 메모리 로드 완료")
    return _llm


def ensure_base_llm():
    global _base_llm
    if _base_llm is None:
        _say(f"[init] Base 모델 다운로드 중: {{BASE_GGUF_REPO_ID}}/{{BASE_GGUF_FILENAME}}")
        p = hf_hub_download(repo_id=BASE_GGUF_REPO_ID, filename=BASE_GGUF_FILENAME)
        _say(f"[init] Base 파일 다운로드 완료: {{p}}")
        _base_llm = Llama(model_path=p, n_ctx=2048, n_threads=2, n_batch=512, verbose=False)
        _say("[init] Base 모델 메모리 로드 완료")
    return _base_llm


def strip_thinking(text: str) -> str:
    if "</think>" in text:
        text = text.split("</think>", 1)[1]
    return text.lstrip()


def chat_fn(message: str, history: list, system_prompt: str, model_choice: str = SFT_CHOICE):
    # --- 모델 선택 ---
    if model_choice == BASE_CHOICE:
        model = ensure_base_llm()
        model_tag = "qwen3-1.7b-base"
    else:
        model = ensure_llm()
        model_tag = "qwen3.5-2b-sft-q4_k_m"

    # --- messages 조립 ---
    messages = []
    sp = (system_prompt or "").strip()
    if sp:
        messages.append({{"role": "system", "content": sp}})
    for turn in history:
        messages.append({{"role": turn["role"], "content": turn["content"]}})
    messages.append({{"role": "user", "content": message}})

    # --- 요청 로그 (즉시 flush) ---
    _say("━" * 70)
    _say(f"[turn-request] model={{model_choice}} tag={{model_tag}}")
    _say(f"[turn-request] system_prompt (len={{len(sp)}}): {{sp[:250]!r}}")
    _say(f"[turn-request] user: {{message!r}}")
    _say(f"[turn-request] messages count={{len(messages)}}")
    for i, m in enumerate(messages):
        c = m.get("content", "")
        preview = c if len(c) <= 250 else c[:250] + "…"
        _say(f"  [{{i}}] role={{m.get('role')!r}} content={{preview!r}}")

    # --- Langfuse observation 시작 ---
    gen = None
    if langfuse is not None:
        with propagate_attributes(tags=DEMO_TAGS + [model_tag]):
            gen = langfuse.start_observation(
                as_type="generation",
                name="gradio-demo-chat",
                model=model_tag,
                input=messages,
                metadata={{
                    "condition": CONDITION,
                    "model_choice": model_choice,
                    "system_prompt_used": bool(sp),
                    "deployment": DEPLOYMENT,
                    "inference_engine": "llama.cpp",
                    "question": message,
                }},
            )

    # --- 스트리밍 생성 ---
    t0 = time.perf_counter()
    buffer = ""
    for chunk in model.create_chat_completion(
        messages=messages,
        temperature=0.7,
        max_tokens=MAX_NEW_TOKENS,
        stop=GEN_STOP_TOKENS,
        stream=True,
    ):
        delta = chunk["choices"][0]["delta"].get("content", "")
        if not delta:
            continue
        buffer += delta
        yield strip_thinking(buffer)

    latency_s = time.perf_counter() - t0
    latency_ms = int(latency_s * 1000)

    try:
        in_tok  = len(model.tokenize((sp + "\\n" + message).encode("utf-8")))
        out_tok = len(model.tokenize(buffer.encode("utf-8")))
    except Exception:
        in_tok, out_tok = 0, 0
    tps = out_tok / latency_s if latency_s > 0 else 0.0

    cleaned = strip_thinking(buffer)
    _say(f"[turn-response] latency_ms={{latency_ms}} in_tok={{in_tok}} out_tok={{out_tok}} tps={{tps:.2f}}/s")
    _say(f"[turn-response] output: {{cleaned[:300]!r}}")
    _say("━" * 70)

    if gen is not None:
        try:
            gen.update(
                output=cleaned,
                usage_details={{"input": in_tok, "output": out_tok, "total": in_tok + out_tok}},
                metadata={{"latency_ms": latency_ms, "tokens_per_sec": round(tps, 2)}},
            )
            gen.end()
        except Exception:
            pass


# ---------- Gradio UI ----------
EXAMPLE_QUESTIONS = [
    ["밤에 사진 찍으면 색이 이상해요"],
    ["폴드 펼칠 때 화면이 깜빡여요"],
    ["배터리가 하루도 안 가요"],
    ["카메라 초기화하는 방법 알려주세요"],
    ["스마트홈 액세서리에서 알림이 안 울려요"],
]

demo = gr.ChatInterface(
    fn=chat_fn,
    type="messages",
    title="쇼핑몰 고객 지원팀 고객 상담 챗봇",
    description=(
        "두 가지를 UI에서 직접 비교할 수 있습니다.\\n\\n"
        "1) **System Prompt 토글** — 03 평가의 조건 3(SFT만) vs 조건 4(SFT + 프롬프트).\\n"
        "2) **모델 선택** — 학습한 **SFT 모델** vs 학습 안 한 **Base(Qwen3-1.7B)**."
    ),
    examples=EXAMPLE_QUESTIONS,
    additional_inputs=[
        gr.Textbox(
            label="System Prompt",
            value=DEFAULT_SYSTEM_PROMPT,
            lines=3,
            info="비워두면 System Prompt 없이 순수 모델 행동으로 답변합니다.",
        ),
        gr.Radio(
            label="모델 선택",
            choices=[SFT_CHOICE, BASE_CHOICE],
            value=SFT_CHOICE,
            info="Base를 처음 선택하면 ~1분간 다운로드+로드 (이후 즉시).",
        ),
    ],
)

_say(f"[boot] UI 준비 완료 | SFT={{GGUF_REPO_ID}} Base={{BASE_GGUF_REPO_ID}}")
_say("[boot] Gradio launch 시작...")
demo.launch(server_name="0.0.0.0", server_port=7860)
'''
(SPACES_DIR / "app.py").write_text(app_py, encoding="utf-8")

# --- README.md ---
readme_md = f"""---
title: Product CS
emoji: \"\\U0001F4F1\"
colorFrom: blue
colorTo: indigo
sdk: docker
pinned: false
license: apache-2.0
---

# Product CS Chatbot

UI에서 **우리 SFT 모델**과 **학습 전 Base(Qwen3-1.7B)** 를 전환해 비교할 수 있습니다.
Container 로그에 매 턴마다 `[turn-request]`/`[turn-response]` 블록이 찍힙니다.
"""
(SPACES_DIR / "README.md").write_text(readme_md, encoding="utf-8")

req_path = SPACES_DIR / "requirements.txt"
if req_path.exists():
    req_path.unlink()

print("✅ Spaces 배포 파일 작성 완료 (로그 강화판)")
for p in sorted(SPACES_DIR.iterdir()):
    print(f"   {p}  ({p.stat().st_size} bytes)")
print(f"\n대상 Space repo: {SPACE_REPO_ID}")
print(f"로그 장치:")
print(f"  - Dockerfile: CMD [\"python\", \"-u\", \"app.py\"] + ENV PYTHONUNBUFFERED=1")
print(f"  - app.py: sys.stdout.reconfigure(line_buffering=True)")
print(f"  - app.py: print+logging 이중 출력 (_say 함수)")
print(f"  - 확인: Spaces → Logs → Container 탭 → [boot] 및 [turn-*] 검색")


---
## Phase 9: Spaces에 업로드 → 자동 배포

방금 만든 파일 3개를 Spaces에 올리면 **HF가 알아서 빌드해서 공개 URL을 만들어 줍니다**. 셀 한 번 실행으로 세 가지가 자동 처리:

1. Space 저장소 생성
2. **Secret(비밀 환경변수) 주입** — `HF_TOKEN`은 04에서 만든 private GGUF를 다운로드할 때 필요, Langfuse 키는 대화 로그 기록용
3. 파일 업로드 → 자동 빌드 시작

**빌드 진행 보는 법**

1. 셀 실행 후 출력된 Spaces URL 클릭
2. 페이지 상단 **Logs 탭 → Build** 서브탭에서 진행 상황 확인 (5~10분 소요)
3. 끝나면 **Container 탭**으로 전환 → `[boot] app.py 시작` 로그가 보이면 완료
4. URL이 살아남 → 채팅 가능

이후 코드를 고쳐 셀 16~18을 다시 실행하면 **이미 설치된 부분은 건너뛰고** 재배포는 수 초로 끝납니다.

In [12]:
from huggingface_hub import HfApi, create_repo

api = HfApi()

# --- 1) Space repo 생성 (Docker SDK) ---
# 기존에 Gradio SDK로 만든 Space가 있으면 SDK를 바꿀 수 없으니
# HF 웹에서 해당 Space를 먼저 삭제하거나 SPACE_REPO_ID를 변경하세요.
create_repo(
    repo_id=SPACE_REPO_ID,
    repo_type="space",
    space_sdk="docker",        # ← Gradio SDK 아님
    private=False,
    exist_ok=True,
)
print(f"✅ Space repo 준비: {SPACE_REPO_ID} (sdk=docker)")

# --- 2) Secret 주입 ---
# HF_TOKEN: 04에서 만든 GGUF repo가 private이므로 런타임 다운로드에 필요.
#           huggingface_hub 라이브러리가 환경변수 HF_TOKEN을 자동으로 읽어 인증 헤더에 부착.
# LANGFUSE_*: observation 기록용 (없으면 앱은 동작하되 Langfuse 기록만 skip).
SECRET_KEYS = [
    "HF_TOKEN",
    "LANGFUSE_PUBLIC_KEY",
    "LANGFUSE_SECRET_KEY",
    "LANGFUSE_BASE_URL",
]
for key in SECRET_KEYS:
    val = os.environ.get(key)
    if not val:
        print(f"⚠️  {key}가 비어있어 Secret 주입을 건너뜁니다.")
        continue
    api.add_space_secret(repo_id=SPACE_REPO_ID, key=key, value=val)
    print(f"✅ Secret 주입: {key}")

# --- 3) 파일 업로드 ---
# Dockerfile이 마지막에 올라가면 Spaces가 자동으로 빌드 시작.
for fname in ["README.md", "app.py", "Dockerfile"]:
    api.upload_file(
        path_or_fileobj=str(SPACES_DIR / fname),
        path_in_repo=fname,
        repo_id=SPACE_REPO_ID,
        repo_type="space",
    )
    print(f"⬆️  업로드: {fname}")

SPACE_URL = f"https://huggingface.co/spaces/{SPACE_REPO_ID}"
print(f"\n✅ 업로드 완료. Spaces가 자동 Docker 빌드를 시작합니다.")
print(f"   Spaces URL:  {SPACE_URL}")
print(f"   빌드 로그:    위 URL → Logs 탭 → Build")
print(f"   예상 시간:   ~5~10분 (첫 빌드), 이후 재배포는 레이어 캐시로 수 초")
print(f"   상태 변화:   Building → Running")

---
## Phase 10: 실 사용 + Langfuse 검증 + 턴별 로그 읽기

### 빌드 완료 후 확인 사항

1. **Spaces URL** — 폰/다른 PC에서 열어 예시 질문 클릭, System Prompt 켜기/끄기, 모델 선택 전환
2. **Container 로그** — Space 페이지 → **Logs 탭 → Container** 에서 매 턴마다 찍히는 `[turn] ...` 로그 확인. 아래 형식으로 나옵니다:
   ```
   [turn] model=Base (Qwen3-1.7B, 학습 안 함) (qwen3-1.7b-base)
   [turn] system_prompt (len=149): '당신은 반말로 답하는 상담원이야. 예시:...'
   [turn] messages (n=2):
     [0] role='system'    content='...'
     [1] role='user'      content='밤에 사진 찍으면 색이 이상해요'
   [turn] latency_ms=34521 in_tok=147 out_tok=84 tps=2.43/s
   [turn] output: '안녕하세요. 밤에 촬영할 때...'
   ```
3. **Langfuse 대시보드** — `tag:gradio-demo` 필터, 모델별 태그(`qwen3.5-2b-sft-q4_k_m` / `qwen3-1.7b-base`)로 세부 비교

### 속도에 대해 — 이건 하드웨어 한계입니다

**cpu-basic의 물리적 상한이 이 정도입니다.**

| 항목 | 값 |
|---|---|
| vCPU | 2개 (shared) |
| RAM | 16GB |
| tokens/sec | **실측 3~8 tok/s** (로그의 `tps=` 수치로 확인) |
| 100 토큰 생성 | 15~30초 |
| 200 토큰 생성 | 30~60초 |
| 첫 턴 (모델 다운로드+로드 포함) | **1분 이상** |

느린 이유는 세 가지가 겹칩니다. 첫째, cpu-basic은 "2 vCPU shared"라 다른 Space와 경쟁합니다. 둘째, Q4_K_M이어도 2B 모델은 매 토큰마다 1.4GB weight를 훑어야 합니다. 셋째, Spaces 무료 티어는 메모리 대역폭도 낮은 편입니다.

**더 빠르게 하는 선택지**:

- **`cpu-upgrade`로 업그레이드**: 8 vCPU + 32GB RAM. 시간당 약 $0.03 (Space가 running일 때만 과금, sleep 중엔 무료). 토큰 속도 3~5배.
- **ZeroGPU**: HF Pro 구독($9/월)이 있으면 공용 A100 접근 가능. GGUF보다 FP16/BF16 safetensors가 더 적합.
- **더 작은 모델**: Qwen3-0.6B-GGUF면 현재 cpu-basic에서도 2~3배 빠름. 품질은 타협.
- **강의 시연용이면 그냥 받아들이기**: "CPU만으로 돈다"가 14강의 메시지. 지연 자체가 증거물.

### Base 모델이 존댓말만 쓰는 이유

로그에서 `system_prompt`가 원본 그대로 전달되고, `messages[0].role=\'system\'`으로 들어간다는 게 확인되면 **코드 문제는 아니라는 신호**입니다. 그런데도 Base가 반말을 안 쓴다면 원인은 다음입니다:

**Qwen3-1.7B-Instruct는 공식 RLHF(Reinforcement Learning from Human Feedback, 사람 피드백으로 강화학습함)를 거친 모델입니다.** 그 RLHF 학습 데이터가 한국어 경우 대부분 정중한 어미(~요, ~니다)로 구성돼 있어요. 그래서:

- 프롬프트에 "반말 써"라고 써도 **RLHF가 심어놓은 정중함 편향**을 완전히 못 이깁니다.
- 우리 SFT 모델은 거기에 추가로 **296개 × 2 epoch의 격식체만** 학습했으니 더 강합니다.
- 그래서 "SFT가 Base보다 더 극단적으로 정중함에 고정"됐다는 게 정확한 해석입니다. "Base가 반말로 잘 답한다"가 아니라 "Base도 약간 덜 극단적일 뿐"입니다.

이전 Colab 테스트에서 Base가 "~게 좋다" 같은 반말을 일부 섞었던 건 temperature=0.7 샘플링의 변동이었습니다. 여러 번 돌리면 가끔은 반말이 섞이고, 가끔은 완전 존댓말만 나옵니다. **RLHF 편향을 완전히 깨려면 프롬프트가 아니라 추가 학습이 필요하다**는 증거 자체는 그대로입니다.

> ⚠️ Langfuse Cloud는 기본으로 배치 전송. `langfuse.flush()`를 호출하면 즉시 대시보드 반영.


In [13]:
# Langfuse 버퍼 플러시 (방금 Colab에서 한 대화들을 대시보드에 즉시 반영)
langfuse.flush()

print("=" * 70)
print("🎯 확인 체크리스트")
print("=" * 70)

print(f"\n[1] Spaces URL (상시 공개, 만료 없음)")
print(f"    {SPACE_URL}")
print(f"    → 폰 브라우저로 열어서 예시 질문 클릭해보세요")

print(f"\n[2] Langfuse 대시보드")
print(f"    {os.environ['LANGFUSE_BASE_URL']}")
print(f"    → Traces 탭 → Filter: tag = 'gradio-demo'")
print(f"    → Colab 대화 + Spaces 대화가 같은 Project에 쌓입니다")

print(f"\n[3] 같은 Project에서 03/04 데이터도 함께 조회")
print(f"    - 03 tag: 'sft-eval'            (평가 trace)")
print(f"    - 04 tag: 'quantization'        (양자화 품질 평가)")
print(f"    - 05 tag: 'gradio-demo'         (실 사용자 대화) ← 지금 추가됨")
print(f"    → 학습→평가→양자화→실 사용까지 한 대시보드에서 시계열로 조회")

---
## 완료 — 전체 파이프라인

| 단계 | 결과물 | 위치 |
|---|---|---|
| 01 데이터 생성 | `train.jsonl`, `test.jsonl` | Drive |
| 02 LoRA SFT 학습 | FP16 merged 모델 | `<user>/product-cs-qwen3.5-2b-sft` |
| 03 4조건 평가 | Langfuse Trace + Judge Score | Langfuse Project |
| 04 GGUF 양자화 | FP16-gguf / Q8_0 / Q4_K_M | `<user>/product-cs-qwen3.5-2b-gguf` |
| **05 Gradio + Spaces** | **공개 데모 URL + 실 사용 Trace + SFT vs Base 비교** | **`<user>/product-cs-chatbot-demo`** |

**오늘 달성한 것**

1. ✅ **CPU만으로 LLM 데모 가동** — 14강 "GGUF가 CPU에서 돌아간다"의 무료 배포 증명
2. ✅ **비개발자용 URL 확보** — "파일 다운로드 받으세요"가 아니라 "링크 클릭하세요"
3. ✅ **평가용 Langfuse가 production 모니터링으로 진화** — 같은 Project, 다른 tag
4. ✅ **System Prompt 토글 = 03 4조건의 라이브 체험** — 숫자로 본 차이를 손으로 확인
5. ✅ **SFT vs Base 모델 비교 실험 UI에 탑재** — 같은 프롬프트를 두 모델에 던져 SFT 각인이 프롬프트를 어떻게 이기는지 직접 확인

**강의 마무리 포인트**

> 같은 프롬프트·같은 질문에 Base는 첫 문장부터 반말을 시도하지만, 우리 SFT는 `안녕하세요!`부터 시작해 끝까지 해요체·격식체만 씁니다. **296개 × 2 epoch이 모델에 박은 각인이 명시적 시스템 프롬프트를 이깁니다.** 행동을 확실히 바꾸려면 프롬프트가 아니라 데이터입니다.

**다음 스텝(숙제)**

- Spaces URL을 팀 슬랙에 공유 → 며칠 뒤 Langfuse에서 실 사용 질문 패턴 분석
- 자주 등장하는 실패 패턴을 `01_synthetic_data`에 추가 → 재학습 → 재배포
- 06에서 만들 (옵션) Agentic RAG와 결합하면 **"재학습 없이 새 도메인 지식 주입"**도 가능